In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import cross_validate
from sklearn.model_selection import cross_val_predict

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    make_scorer
)

In [2]:
feature_df = pd.read_csv(
    r"C:\Users\aashi\Desktop\FYP\EDA\parkinson_features.csv"
)

feature_df.head()

print(feature_df.shape)

feature_df.info()

(195, 10)
<class 'pandas.DataFrame'>
RangeIndex: 195 entries, 0 to 194
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   data_file_name  195 non-null    str    
 1   updrs           195 non-null    int64  
 2   age             195 non-null    int64  
 3   gender          195 non-null    str    
 4   patient_off_on  195 non-null    str    
 5   speed           195 non-null    float64
 6   amplitude       195 non-null    float64
 7   hesitation      195 non-null    float64
 8   halt            195 non-null    float64
 9   decrement       195 non-null    float64
dtypes: float64(5), int64(2), str(3)
memory usage: 15.4 KB


In [3]:
df = feature_df.copy()

#encode gender
gender_encoder = LabelEncoder()

df["gender"] = gender_encoder.fit_transform(df["gender"])

In [4]:
#encode medication
medication_encoder = LabelEncoder()

df["patient_off_on"] = medication_encoder.fit_transform(
    df["patient_off_on"]
)

In [5]:
#separate features and labels
X = df.drop(
    columns=[
        "data_file_name",
        "updrs"
    ]
)

y = df["updrs"]

In [6]:
#feature scaling for logistic regression and SVM

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [7]:
#cross validation
cv = RepeatedStratifiedKFold(

    n_splits=5,

    n_repeats=10,

    random_state=42

)

In [8]:
print(X.shape)

print(y.shape)

print(X.columns)

(195, 8)
(195,)
Index(['age', 'gender', 'patient_off_on', 'speed', 'amplitude', 'hesitation',
       'halt', 'decrement'],
      dtype='str')


In [9]:
# =====================================================
# Evaluation Metrics
# =====================================================

scoring = {

    "accuracy": "accuracy",

    "precision": make_scorer(
        precision_score,
        average="macro",
        zero_division=0
    ),

    "recall": make_scorer(
        recall_score,
        average="macro",
        zero_division=0
    ),

    "f1": make_scorer(
        f1_score,
        average="macro",
        zero_division=0
    )

}

In [10]:
# =====================================================
# Generic Evaluation Function
# =====================================================

def evaluate_model(model, model_name):

    print("=" * 70)
    print(model_name)
    print("=" * 70)

    # --------------------------------------------
    # Cross Validation Scores
    # --------------------------------------------

    scores = cross_validate(

        estimator=model,

        X=X_scaled,

        y=y,

        cv=cv,

        scoring=scoring,

        n_jobs=-1,

        return_train_score=False

    )

    # --------------------------------------------
    # Mean Scores
    # --------------------------------------------

    accuracy = scores["test_accuracy"].mean()
    precision = scores["test_precision"].mean()
    recall = scores["test_recall"].mean()
    f1 = scores["test_f1"].mean()

    # --------------------------------------------
    # Standard Deviations
    # --------------------------------------------

    acc_std = scores["test_accuracy"].std()
    f1_std = scores["test_f1"].std()

    print(f"Accuracy : {accuracy:.4f} ± {acc_std:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"Macro F1 : {f1:.4f} ± {f1_std:.4f}")

    # --------------------------------------------
    # Cross-Validated Predictions
    # --------------------------------------------

    predictions = cross_val_predict(

        estimator=model,

        X=X_scaled,

        y=y,

        cv=5,

        n_jobs=-1

    )

    # --------------------------------------------
    # Classification Report
    # --------------------------------------------

    print("\nClassification Report\n")

    print(

        classification_report(
            y,
            predictions,
            zero_division=0
        )

    )

    # --------------------------------------------
    # Confusion Matrix
    # --------------------------------------------

    cm = confusion_matrix(
        y,
        predictions
    )

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm
    )

    fig, ax = plt.subplots(figsize=(6,6))

    disp.plot(
        ax=ax,
        cmap="Blues",
        colorbar=False
    )

    plt.title(model_name)

    plt.show()

    # --------------------------------------------
    # Return Results
    # --------------------------------------------

    return {

        "Model": model_name,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "Macro F1": f1

    }